---
title: Beginner's Guide exercise solution
short_title: Exercise solution
subject: Beginner Guide
subtitle: A worked solution for the Beginner's Guide exercise.
description: A worked solution for the Beginner's Guide exercise.
keywords:
  - open-data-cube
  - odc
  - xarray
  - plotting
  - spectral-indices
  - beginner-guide
  - exercise
---

This notebook gives a worked solution to the Beginner's Guide exercise using a point near Kuta, Lombok.[^edits]
Each step explains the reason for the operation and what to look for in its output.

[^edits]: Tutorial notebooks update automatically; edits to a tutorial notebook may be overwritten on the next update.
    Keep a working copy in a separate file to preserve changes.

## A. Objectives

- Turn a point into an area of interest
- Find and load annual satellite data
- Inspect an `xarray.Dataset`
- Plot a true-colour image
- Calculate and plot NDVI and NDWI
- Compare annual index plots

## B. Preparation

`Datacube` opens the data connection, while `point` creates a spatial geometry from a longitude and latitude.
The application name identifies this connection in datacube logs; assigning it to `dc` makes the same connection available to the later query.

In [ ]:
from datacube import Datacube
from odc.geo.geom import point

dc = Datacube(app="beginners_guide_exercise")

This cell produces no visible result because the connection is stored for later use.
A connection error at this stage usually means the datacube environment is unavailable, so it should be resolved before continuing.

## C. Data loading

A point has no area, so it must first be expanded before it can define a satellite data request.
The `0.05`-degree buffer creates that area, and `boundingbox` converts it into the left, right, bottom, and top bounds required by the query.

In [ ]:
latitude = -8.81
longitude = 116.008667

bbox = point(longitude, latitude, crs="EPSG:4326").buffer(0.05).boundingbox
bbox.explore()

The map provides a quick spatial check before any data are loaded.
Its rectangle should enclose the area around Kuta; an unexpected location would usually indicate that longitude and latitude had been reversed.

The query requests the annual Sentinel-2 GeoMAD product from 2022 to 2025, including the four measurements needed for the true-colour image and spectral indices.
Kuta lies in UTM zone 50S, so `EPSG:32750` gives the output grid metre-based coordinates and supports the requested 30-metre resolution.

In [ ]:
query = {
    "product": "s2_geomad_annual",
    "x": (bbox.left, bbox.right),
    "y": (bbox.bottom, bbox.top),
    "time": ("2022", "2025"),
    "measurements": ["red", "green", "blue", "nir"],
    "output_crs": "EPSG:32750",
    "resolution": (-30, 30),
}

ds = dc.load(**query)

The loaded data are stored in `ds` as an `xarray.Dataset`.
No output appears from the assignment itself; the following cells inspect what the query returned.

## D. Dataset inspection

Displaying the Dataset first gives an overview of its dimensions, coordinates, measurements, and attributes.
The remaining expressions isolate the details requested in the exercise.

In [ ]:
ds.sizes

The size mapping reports the length of each dimension.
The `time` dimension should contain four annual slices, while `x` and `y` report the number of 30-metre pixels across the area.

In [ ]:
ds.data_vars

The data variables should include `red`, `green`, `blue`, and `nir`.
Each variable uses the same `time`, `y`, and `x` dimensions, which allows the bands to be combined pixel by pixel.

In [ ]:
ds.attrs["crs"]

The CRS output should identify `EPSG:32750`, matching the output grid requested in the query.

## E. True-colour image

A true-colour image needs the red, green, and blue measurements in one array.
`to_array` stacks them along a new `band` dimension, and `isel(time=0)` selects the first annual composite.
The display range of 0 to 3000 prevents a few bright pixels from setting the contrast for the whole image.

In [ ]:
rgb = ds[["red", "green", "blue"]].to_array(dim="band").isel(time=0)
rgb.plot.imshow(vmin=0, vmax=3000)

The resulting colours provide a familiar reference for the index plots.
Vegetated land usually appears green, open water dark, and built-up or bare surfaces lighter, although an annual composite will not look exactly like a single-date photograph.

## F. Spectral index calculation

NDVI compares near-infrared and red reflectance to emphasise vegetation, while NDWI compares green and near-infrared reflectance to emphasise water.
The source bands are converted to floating-point values before subtraction and division. This avoids integer and unsigned-arithmetic artefacts and ensures the indices are calculated as floating-point values.

$$\text{NDVI} = \frac{\text{NIR} - \text{Red}}{\text{NIR} + \text{Red}}$$

$$\text{NDWI} = \frac{\text{Green} - \text{NIR}}{\text{Green} + \text{NIR}}$$

In [ ]:
red = ds.red.astype("float32")
green = ds.green.astype("float32")
nir = ds.nir.astype("float32")

ds["ndvi"] = (nir - red) / (nir + red)
ds["ndwi"] = (green - nir) / (green + nir)

ds[["ndvi", "ndwi"]]

The output confirms that `ndvi` and `ndwi` have been added to the Dataset for every time slice and pixel.
Most valid values lie between -1 and 1; missing values remain missing where the source data do not support a calculation.

## G. Index plots

A fixed scale from -1 to 1 keeps each colour tied to the same value across all plots.
Without that shared scale, similar colours in separate panels could represent different index values.

In [ ]:
ds.ndvi.isel(time=0).plot(cmap="RdYlGn", vmin=-1, vmax=1)

In the NDVI plot, greener pixels have higher values and usually indicate denser or healthier vegetation.
Yellow and red pixels have lower values and are more likely to be sparse vegetation, bare or built-up land, or water.

In [ ]:
ds.ndwi.isel(time=0).plot(cmap="RdBu", vmin=-1, vmax=1)

In the NDWI plot, blue pixels have higher values and commonly correspond to open water.
Red pixels have lower values and usually correspond to land.

In [ ]:
ds.ndvi.plot(col="time", col_wrap=2, cmap="RdYlGn", vmin=-1, vmax=1)

The facet grid places one NDVI map beside another and labels each panel by time.
Because every panel uses the same extent and scale, a colour change at the same location indicates a change in the annual composite rather than a change in plotting limits.

## H. Result interpretation

1. The highest NDVI values occur in the greener, densely vegetated parts of the land.
   The lowest values occur over the sea and in bare or built-up areas around Kuta.
2. Open-water areas generally combine high NDWI with low NDVI.
   This opposite response follows from strong water absorption in the near-infrared band, which lowers NDVI while raising NDWI.
3. The clearest annual changes appear where land pixels shift most strongly between red, yellow, and green in the NDVI panels.
   Patches around cultivated, cleared, or developing land tend to vary more than persistent open water or stable dense vegetation.
4. Some differences may represent land-cover or vegetation change.
   Others may come from seasonal conditions and from the set of valid observations combined into each annual GeoMAD composite, so a changed colour alone does not prove a permanent change on the ground.

The true-colour image helps distinguish water, vegetation, bare ground, and built-up surfaces before these explanations are assigned to the index patterns.

## I. Optional challenge

The same facet layout can be applied to NDWI.
Comparing it with the NDVI grid shows whether locations that gain or lose vegetation also change in their water response.

In [ ]:
ds.ndwi.plot(col="time", col_wrap=2, cmap="RdBu", vmin=-1, vmax=1)

Here, a shift towards blue means higher NDWI, while a shift towards red means lower NDWI.
The fixed scale again makes those shifts comparable from year to year.

## J. Next steps

Restart the kernel and run all cells to confirm that the solution has no hidden state.
Once it runs without errors, compare each section with the exercise and note where another valid approach could produce the same result.